# Inference for Regression

In [1]:
library(tidyverse)
library(infer)
library(broom)

options(repr.matrix.max.rows = 20)
set.seed(42)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


## Setup

We'll use the `baby` dataset throughout. Our main question: what predicts **Birth Weight**?

In [ ]:
baby <- read_csv("baby.csv", show_col_types = FALSE)
glimpse(baby)

## Slope Uncertainty: Observed Slope

First, compute the observed slope using the correlation identity: $b_1 = r \cdot \frac{s_y}{s_x}$

In [ ]:
summary <- baby |>
  summarize(
    r     = cor(`Gestational Days`, `Birth Weight`),
    slope = r * sd(`Birth Weight`) / sd(`Gestational Days`)
  )
summary

## Slope Uncertainty: Bootstrap Distribution

We resample rows with replacement 5000 times, refit `lm()` each time, and collect the slope.

In [ ]:
set.seed(123)
boot_dist <- baby |>
  specify(`Birth Weight` ~ `Gestational Days`) |>
  generate(reps = 5000, type = "bootstrap") |>
  calculate(stat = "slope")

boot_dist |>
  ggplot(aes(x = stat)) +
  geom_histogram(bins = 20, fill = "chartreuse4", color = "white") +
  labs(x = "Bootstrap Slopes", y = "Count") +
  theme_minimal(base_size = 16)

## Slope Uncertainty: 95% Confidence Interval

In [ ]:
ci <- boot_dist |>
  get_ci(level = 0.95, type = "percentile")
ci

In [ ]:
left      <- ci$lower_ci
right     <- ci$upper_ci
obs_slope <- summary$slope

boot_dist |>
  ggplot(aes(x = stat)) +
  geom_histogram(bins = 20, fill = "chartreuse4", color = "white") +
  geom_vline(xintercept = c(left, right), linetype = "dashed", linewidth = 1) +
  geom_vline(xintercept = obs_slope, linewidth = 1) +
  labs(
    title = "Bootstrap Distribution of Slopes",
    x = "Bootstrap Slopes",
    y = "Count",
    subtitle = paste0(
      "Observed slope = ", signif(obs_slope, 4),
      " | 95% CI = [", signif(left, 4), ", ", signif(right, 4), "]"
    )
  ) +
  theme_minimal(base_size = 16)

## Try It Out!

What other variables predict Birth Weight? Compute bootstrap CIs for each predictor below.

**Null hypothesis:** The slope is 0 (no linear relationship).  
**Alternative hypothesis:** The slope is not 0.

If the 95% CI does **not** include 0, we have evidence against the null.

In [ ]:
names(baby)

### 1. Maternal Age

### 2. Maternal Pregnancy Weight

### 3. Maternal Smoker

We first convert `Maternal Smoker` from logical to numeric (TRUE → 1, FALSE → 0).

## Prediction Uncertainty at a Chosen x

We now bootstrap **predicted values** $\hat{y}$ at $x_0 = 300$ gestational days.

For each bootstrap resample, we refit `lm()` and collect the predicted birth weight at 300 days.

In [ ]:
x0 <- 300

set.seed(123)
boot_dist <- baby |>
  specify(`Birth Weight` ~ `Gestational Days`) |>
  generate(reps = 5000, type = "bootstrap") |>
  group_by(replicate) |>
  nest() |>
  mutate(
    model = map(data, ~ lm(`Birth Weight` ~ `Gestational Days`, data = .x)),
    yhat  = map_dbl(model, ~ predict(.x, newdata = tibble(`Gestational Days` = x0)))
  ) |>
  select(replicate, yhat)

glimpse(boot_dist)

In [ ]:
boot_dist |>
  ggplot(aes(x = yhat)) +
  geom_histogram(bins = 20, fill = "chartreuse4", color = "white") +
  labs(
    x = "Bootstrap predictions at x = 300",
    y = "Count",
    title = "Prediction variability via bootstrap"
  ) +
  theme_minimal(base_size = 16)

In [ ]:
ci <- boot_dist |>
  rename(stat = yhat) |>
  get_ci(level = 0.95, type = "percentile")
ci

## CI vs. PI: Comparing Intervals at x = 300

Using `predict()` directly gives us both a **confidence interval** (for the mean response) and a **prediction interval** (for a new individual).

In [ ]:
x0 <- 300
model <- lm(`Birth Weight` ~ `Gestational Days`, data = baby)

predict(model, newdata = tibble(`Gestational Days` = x0))

predict(model, newdata = tibble(`Gestational Days` = x0),
        interval = "confidence", level = 0.95)

predict(model, newdata = tibble(`Gestational Days` = x0),
        interval = "prediction", level = 0.95)

## Visual: Fitted Line + Prediction at x = 300

In [ ]:
fit     <- lm(`Birth Weight` ~ `Gestational Days`, data = baby)
fit_300 <- predict(fit, newdata = tibble(`Gestational Days` = 300))

ggplot(baby, aes(`Gestational Days`, `Birth Weight`)) +
  geom_point(alpha = 0.5, color = "chartreuse4") +
  geom_smooth(method = "lm", se = FALSE, color = "goldenrod2", linewidth = 1) +
  geom_segment(aes(x = 300, xend = 300, y = 0, yend = fit_300),
               color = "red", linewidth = 1.1) +
  labs(
    title = "Fitted line and prediction at x = 300",
    x = "Gestational Days",
    y = "Birth Weight"
  ) +
  theme_minimal(base_size = 16)

## Residual Diagnostics

Before trusting our intervals, we check whether the linear model assumptions are reasonable.

If residuals look randomly scattered around 0, the model is appropriate. Patterns mean something is being missed.

In [ ]:
fit <- lm(`Birth Weight` ~ `Gestational Days`, data = baby)
aug <- augment(fit)

ggplot(aug, aes(.fitted, .resid)) +
  geom_hline(yintercept = 0, linetype = "dashed") +
  geom_point(alpha = 0.5) +
  labs(title = "Residuals vs Fitted", x = "Fitted", y = "Residuals") +
  theme_minimal(base_size = 16)

## Try It Out!

- Compute the bootstrap distribution for the prediction at **285 gestational days**.
- Compute the confidence interval.
- Compute the prediction interval.
- Compare your results to what we got at 300 days — which is wider? Why?

In [ ]:
# your code here

## Discussion: Predictions Near vs. Far from the Mean

Why does prediction variability depend on how far $x_0$ is from the mean of $x$?

In [ ]:
mean(baby$`Gestational Days`)